# RNN for Time Series Prediction — Edge AI

**RNN implementation for time series prediction using PyTorch — built as preparation for Edge AI research.**

Inspired by: Tokgoz et al. (2023) — *An Integer-Only Resource-Minimized RNN on FPGA for Low-Frequency Sensors in Edge-AI*, IEEE Sensors Journal

---
This notebook demonstrates how a Recurrent Neural Network (RNN) can learn temporal patterns in sequential data.
The core principle — using RNNs for efficient time-series prediction — directly relates to Edge AI applications
where models must run under strict power and memory constraints.

## Step 1: Import Libraries

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# numpy  - for data generation and array operations
# torch  - PyTorch deep learning framework
# nn     - neural network modules (RNN, Linear layers, loss functions)
# matplotlib - for visualizing data and training results

## Step 2: Generate Sine Wave Data

We use a sine wave as our time series dataset.
- **Why sine wave?** It has a clear repeating pattern — perfect for teaching an RNN to learn temporal dependencies.
- **Real-world analogy:** Sensor data (accelerometer, temperature, behavior signals) behaves similarly — repeating patterns over time.
- In Tokgoz's research, the RNN processes sensor data sample-by-sample, predicting the next state.

In [ ]:
np.random.seed(42)  # For reproducibility

steps = 1000  # Total number of data points
x = np.sin(np.linspace(0, 50, steps))  # Generate sine wave: 1000 points from 0 to 50

# Visualize first 100 points to understand the pattern
plt.plot(x[:100])
plt.title("Sine Wave - First 100 points")
plt.xlabel("Time step")
plt.ylabel("Amplitude")
plt.show()

print("Data shape:", x.shape)

## Step 3: Create Sequences

RNNs learn from sequences — they look at the past N timesteps to predict the next value.
- `seq_length = 20` means: the RNN sees 20 previous points and predicts the 21st.
- This is exactly how Tokgoz's architecture works — processing input sample-by-sample using temporal memory.

In [ ]:
def create_sequences(data, seq_length):
    """Convert raw time series into (input_sequence, target) pairs."""
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])   # Input: 20 past values
        y.append(data[i+seq_length])      # Target: the next value to predict
    return np.array(X), np.array(y)

seq_length = 20  # RNN looks back 20 timesteps to make a prediction
X, y = create_sequences(x, seq_length)

print("X shape:", X.shape)  # (980 samples, 20 timesteps each)
print("y shape:", y.shape)  # (980 targets)

## Step 4: Prepare PyTorch Tensors & Train/Test Split

- Convert numpy arrays to PyTorch tensors (required for model training).
- `unsqueeze(-1)` adds a feature dimension: shape becomes (samples, timesteps, features).
- 80% data for training, 20% for testing — standard ML practice.

In [ ]:
# Convert to PyTorch tensors
X_tensor = torch.FloatTensor(X).unsqueeze(-1)  # Shape: (980, 20, 1) — 1 feature per timestep
y_tensor = torch.FloatTensor(y).unsqueeze(-1)  # Shape: (980, 1)

# Train/test split (80% train, 20% test)
split = int(0.8 * len(X_tensor))
X_train, X_test = X_tensor[:split], X_tensor[split:]
y_train, y_test = y_tensor[:split], y_tensor[split:]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

## Step 5: Define the RNN Model

Architecture:
- **RNN Layer:** Processes the input sequence step-by-step, maintaining a hidden state (memory).
  - `input_size=1`: one feature per timestep
  - `hidden_size=32`: 32 internal neurons (the memory capacity)
  - `num_layers=2`: two stacked RNN layers for deeper pattern learning
- **Linear (FC) Layer:** Maps the final hidden state to a single prediction value.

**Connection to Edge AI research:** Tokgoz's paper minimizes RNN resource usage on FPGAs by using
integer-only arithmetic and time-multiplexed hardware — this software implementation explores the
same temporal reasoning principles.

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=2, output_size=1):
        super(RNNModel, self).__init__()
        
        # RNN layer: learns temporal dependencies in the sequence
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        
        # Fully connected layer: maps RNN output to final prediction
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Pass input through RNN — 'out' contains outputs at every timestep
        out, _ = self.rnn(x)
        
        # Use only the last timestep's output for prediction
        out = self.fc(out[:, -1, :])
        return out

model = RNNModel()
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

## Step 6: Train the Model

- **Loss function (MSELoss):** Measures the average squared difference between predicted and actual values.
- **Optimizer (Adam):** Adjusts model weights to minimize the loss — learning rate 0.001.
- We train for 100 epochs, printing loss every 10 epochs to monitor convergence.

In [ ]:
# Loss function: Mean Squared Error — measures prediction accuracy
criterion = nn.MSELoss()

# Optimizer: Adam — adapts learning rate for each parameter
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 100
losses = []

for epoch in range(epochs):
    model.train()          # Set model to training mode
    optimizer.zero_grad()  # Clear gradients from previous step
    
    output = model(X_train)          # Forward pass: compute predictions
    loss = criterion(output, y_train)  # Compute loss
    loss.backward()        # Backward pass: compute gradients
    optimizer.step()       # Update model weights
    
    losses.append(loss.item())

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/100, Loss: {loss.item():.6f}")

# Plot training loss curve
plt.plot(losses)
plt.title("Training Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.show()

## Step 7: Evaluate — Actual vs Predicted

We test the trained model on unseen data (test set) and compare predictions to ground truth.

In [ ]:
model.eval()  # Set model to evaluation mode (disables dropout etc.)

with torch.no_grad():  # No gradient computation needed during inference
    predicted = model(X_test)

# Plot actual vs predicted values
plt.figure(figsize=(12, 4))
plt.plot(y_test.numpy(), label='Actual', color='blue')
plt.plot(predicted.numpy(), label='Predicted', color='orange', linestyle='--')
plt.title('RNN Predictions vs Actual Values')
plt.xlabel('Test Sample')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

# Final test accuracy
mse = criterion(predicted, y_test)
print(f"Test MSE: {mse.item():.6f}")
print("Lower MSE = better prediction accuracy")